# 🧭 Autogen 멀티 에이전트 여행 플래너: Planner × Local × Language × Summary

이 노트북은 **4개의 에이전트**가 역할을 분담해
**3일간의 서울 여행 계획**을 협업으로 작성하고,
**요약 에이전트가 완성본을 제출하며 "TERMINATE"** 로 종료하는 예제입니다.

---

## 🧩 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **planner_agent** | 여행 일정의 골격/루트/테마 설계 |
| **local_agent** | 현지 체험/로컬 장소/맛집 등 디테일 보강 |
| **language_agent** | 언어/커뮤니케이션 팁, 표현/매너 보완 |
| **travel_summary_agent** | 모든 제안 통합, **완성본 제출 + "TERMINATE"** |
| **RoundRobinGroupChat** | 에이전트 순차 실행(오케스트레이션) |
| **TextMentionTermination("TERMINATE")** | 종료 텍스트 등장 시 종료 |
| **Console** | 스트리밍 메시지를 보기 좋게 출력 |

---

## ⚙️ 동작 흐름

1. **Planner**가 초안을 제시  
2. **Local**이 로컬 관점(현지 추천)으로 보강  
3. **Language**가 언어/커뮤니케이션 팁 제공  
4. **Summary**가 통합·완성본 작성 → **"TERMINATE"** 출력 → 종료

---

## 🚀 확장 아이디어

- 예산·교통·식단 등 **전문 에이전트 추가**
- **툴 연동**: 항공/숙소/지도/날씨/환율 API
- **메모리 연결**: 사용자 선호·과거 일정 반영
- **종료 조건 복합화**: 최대 턴수·승인 토큰·품질 점수 임계치

---


In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console

from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()

api_version = os.getenv("AZURE_OPENAI_API_VERSION")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
azure_openai_chat_completion_client = AzureOpenAIChatCompletionClient(
            model=deployment_name,
            azure_endpoint=azure_endpoint,
            api_version=api_version,
            api_key=api_key,
        )



In [2]:
planner_agent = AssistantAgent(
    "planner_agent",
    model_client=azure_openai_chat_completion_client,
    description="A helpful assistant that can plan trips.",
    system_message="You are a helpful assistant that can suggest a travel plan for a user based on their request.",
)

local_agent = AssistantAgent(
    "local_agent",
    model_client=azure_openai_chat_completion_client,
    description="A local assistant that can suggest local activities or places to visit.",
    system_message="You are a helpful assistant that can suggest authentic and interesting local activities or places to visit for a user and can utilize any context information provided.",
)

language_agent = AssistantAgent(
    "language_agent",
    model_client=azure_openai_chat_completion_client,
    description="A helpful assistant that can provide language tips for a given destination.",
    system_message="You are a helpful assistant that can review travel plans, providing feedback on important/critical tips about how best to address language or communication challenges for the given destination. If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale.",
)

travel_summary_agent = AssistantAgent(
    "travel_summary_agent",
    model_client=azure_openai_chat_completion_client,
    description="A helpful assistant that can summarize the travel plan.",
    system_message="You are a helpful assistant that can take in all of the suggestions and advice from the other agents and provide a detailed final travel plan. You must ensure that the final plan is integrated and complete. YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN. When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.",
)

In [3]:
termination = TextMentionTermination("TERMINATE")
group_chat = RoundRobinGroupChat(
    [planner_agent, local_agent, language_agent, travel_summary_agent], termination_condition=termination
)
await Console(group_chat.run_stream(task="3일간의 서울 여행 계획짜주세요."))

---------- TextMessage (user) ----------
3일간의 서울 여행 계획짜주세요.
---------- TextMessage (planner_agent) ----------
물론입니다! 3일간의 서울 여행 계획을 아래와 같이 제안드립니다.

---

### 1일차: 전통과 역사를 느끼는 하루
- **경복궁** (오전)
  - 조선 왕조의 대표 궁궐 탐방
  - 근정전, 경회루, 수정전 등 관람
  - 오전 10시에 진행되는 수문장 교대식 관람 추천

- **북촌 한옥마을** (오후)
  - 전통 한옥들이 모여 있는 마을 산책
  - 카페나 전통찻집 방문 가능

- **인사동** (저녁)
  - 전통 공예품, 기념품 쇼핑
  - 전통 음식점이나 한식당에서 저녁식사

---

### 2일차: 현대와 문화 체험
- **N서울타워** (오전)
  - 남산에서 서울 전경 감상
  - 케이블카 타고 오르기 추천

- **명동** (오후)
  - 쇼핑과 길거리 음식 체험
  - 화장품, 패션 아이템 구매

- **홍대 거리** (저녁)
  - 젊음의 거리, 다양한 카페, 바, 공연장 탐방
  - 맛집에서 저녁 식사 및 라이브 공연 감상 가능

---

### 3일차: 자연과 휴식
- **한강공원** (오전)
  - 자전거 대여해 한강변 라이딩
  - 산책하거나 피크닉

- **동대문 디자인 플라자(DDP)** (오후)
  - 현대 건축과 전시 관람
  - 동대문 시장에서 쇼핑

- **청계천** (저녁)
  - 도심 속 개천 산책
  - 인근 맛집에서 저녁 식사

---

필요 시 교통카드 T-money 구입과 서울 지하철 및 버스 이용을 추천드립니다. 즐거운 여행 되세요!
---------- TextMessage (local_agent) ----------
서울 3일 일정 추천드립니다!

1일차: 전통과 문화 탐방  
- 오전: 경복궁 방문, 수문장 교대식 감상  
- 점심: 근처 북촌 한옥마을 내 전통 찻집이나 한식당  
- 오후: 북촌 한옥마을

TaskResult(messages=[TextMessage(id='10bca59a-6094-4ae2-abce-d60563cb0416', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 14, 53, 555956, tzinfo=datetime.timezone.utc), content='3일간의 서울 여행 계획짜주세요.', type='TextMessage'), TextMessage(id='f68b2e6f-eb5c-427e-8b10-a65f7526b2db', source='planner_agent', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=452), metadata={}, created_at=datetime.datetime(2025, 10, 17, 15, 15, 10, 468013, tzinfo=datetime.timezone.utc), content='물론입니다! 3일간의 서울 여행 계획을 아래와 같이 제안드립니다.\n\n---\n\n### 1일차: 전통과 역사를 느끼는 하루\n- **경복궁** (오전)\n  - 조선 왕조의 대표 궁궐 탐방\n  - 근정전, 경회루, 수정전 등 관람\n  - 오전 10시에 진행되는 수문장 교대식 관람 추천\n\n- **북촌 한옥마을** (오후)\n  - 전통 한옥들이 모여 있는 마을 산책\n  - 카페나 전통찻집 방문 가능\n\n- **인사동** (저녁)\n  - 전통 공예품, 기념품 쇼핑\n  - 전통 음식점이나 한식당에서 저녁식사\n\n---\n\n### 2일차: 현대와 문화 체험\n- **N서울타워** (오전)\n  - 남산에서 서울 전경 감상\n  - 케이블카 타고 오르기 추천\n\n- **명동** (오후)\n  - 쇼핑과 길거리 음식 체험\n  - 화장품, 패션 아이템 구매\n\n- **홍대 거리** (저녁)\n  - 젊음의 거리, 